In [6]:
%pip install -q "vegafusion[embed]>=1.5.0" "vegafusion>=1.5.0" "vl-convert-python>=1.6.0"
%pip install pandas
%pip install altair
%pip install duckdb
%pip install requests
%pip install --upgrade certifi
%pip install --upgrade requests urllib3
%pip install python-certifi-win32
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import altair as alt
import duckdb
import requests
import certifi
import json
import os

In [2]:
alt.data_transformers.enable('vegafusion')
alt.renderers.enable('colab')

RendererRegistry.enable('colab')

In [3]:
response = requests.get("https://data360api.worldbank.org/data360/indicators?datasetId=WB_WDI")
response.status_code

200

In [3]:
def get_wb_data(indicator_id):
    db = "_".join(indicator_id.split("_")[:2])
    base_url = "https://data360api.worldbank.org/data360/data"
    headers = {"accept": "*/*"}

    all_results = []
    skip = 0
    top = 1000

    r = requests.get(
        base_url,
        params={"DATABASE_ID": db, "INDICATOR": indicator_id},
        headers=headers
    ).json()

    total = r.get("count", 0)
    all_results.extend(r.get("value", []))
    print(f"  {indicator_id}: total={total}")

    if len(all_results) >= total:
        pass
    else:
        skip += top
        while skip < total:
            r = requests.get(
                base_url,
                params={"DATABASE_ID": db, "INDICATOR": indicator_id, "skip": skip, "top": top},
                headers=headers
            ).json()
            chunk = r.get("value", [])
            if not chunk:
                break
            all_results.extend(chunk)
            skip += top
            print(f"  {indicator_id}: {min(skip, total)}/{total}")

    if not all_results:
        return pd.DataFrame()

    return pd.json_normalize(all_results).rename(columns={
        "REF_AREA":    "iso3",
        "TIME_PERIOD": "year",
        "OBS_VALUE":   "val",
    })

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/data_cache')
else:
    DATA_DIR = Path(os.getenv("DATA_DIR"))

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_DIR)

C:/Users/u632113/itba_world_bank_grupo2/data_cache
C:\Users\u632113\itba_world_bank_grupo2\data_cache


In [4]:
INDICATORS = {
    "gasto_edu": "WB_WDI_SE_XPD_TOTL_GD_ZS",
    "pib_pc":    "WB_WDI_NY_GDP_PCAP_CD",
    "prim":      "WB_GS_SE_PRM_ENRR",
    "sec":       "WB_GS_SE_SEC_ENRR",
    "ter":       "WB_GS_SE_TER_ENRR",
    "comp":      "WB_GS_SE_PRM_CMPT_ZS",
}

def load_or_fetch(name, indicator_id):
    path = os.path.join(DATA_DIR, f"{name}.parquet")
    if os.path.exists(path):
        print(f"  {name}: cargando desde disco")
        return pd.read_parquet(path)
    print(f"  {name}: descargando...")
    df = get_wb_data(indicator_id)
    df.to_parquet(path, index=False)
    return df

gasto_edu, pib_pc, prim, sec, ter, comp = [
    load_or_fetch(name, ind) for name, ind in INDICATORS.items()
]

  gasto_edu: descargando...


NameError: name 'requests' is not defined